# SmartHome-Bench: 분수 입수와 소파 올라가기 횟수 검증

[SmartHome-Bench](https://huggingface.co/datasets/violetcliff/SmartHome-Bench)의 `smartbench_0642`, `smartbench_0518`을 비교합니다. 두 영상은 서로 다른 장면이며 원본 영상은 로컬 `data/videos/`에만 저장했습니다. 저장된 모델 응답을 읽으므로 이 노트북 실행에는 API 호출이 없습니다.

**주의:** 사람이 센 2회는 0.25~0.5초 프레임을 바탕으로 한 잠정 검토입니다. 사용자의 독립적인 시각 확인 전에는 확정 정답으로 표시하지 않습니다.


## 비교 결과

| 영상 | 행동 | 프레임 잠정 검토 | 현재 일반 추출기 | 시간 전환 전용 VLM (`gpt-4o`) |
|---|---|---:|---:|---:|
| `0518` | 검은 개·흰 개가 소파에 각각 올라감 | **2회** | 명시 1구간(18~22초에 뭉침) | **2건**: 19~20초, 21~21.5초 |
| `0642` | 검은 개 두 마리가 차례로 분수에 들어감 | **2회** | 입수 명시 0건 | 원시 **3행** → 동일 개체의 연속 입수 중복 1행 제외 후 **2회** |

`0642` 모델은 두 번째 입수를 3.5~4.5초와 5~5.5초에 두 번 기록했습니다. 영상에서는 그 사이 같은 개가 물 밖으로 나와 다시 들어간 장면이 보이지 않아 한 행동으로 취급합니다. 이는 사람의 재감사 판정이며 모델의 원시 정답으로 표시하지 않습니다.


In [ ]:
import json
from pathlib import Path
root=Path.cwd()
if root.name == "notebooks": root=root.parent
evidence=root/"notebooks"/"smarthome-bench-evidence"
review=json.loads((evidence/"smartbench-0518-0642-review.json").read_text())
for item in review["videos"]:
    print(item["video_id"], item["question"], "프레임 검토:", item["provisional_count"], "회")
    print("검토 구간:", [(x["subject"],x["start_time"],x["end_time"]) for x in item["intervals"]])


## 원본 영상 재생

로컬 Jupyter에서 아래 셀을 실행하면 두 영상을 직접 볼 수 있습니다. 영상이 없다면 `review.json`의 원본 YouTube 주소로 확인합니다.


In [ ]:
from IPython.display import Video, display
for item in review["videos"]:
    path=root/item["local_video"]
    print(item["video_id"], path if path.exists() else item["source_url"])
    if path.exists(): display(Video(filename=str(path), embed=False, html_attributes="controls"))


## 모델이 실제로 반환한 시각

일반 추출기는 2초 간격 대표 프레임을 사용했습니다. 전환 전용 시험은 전체 영상을 0.5초 간격으로 보고 5초씩 나눠, **행동 전과 후가 둘 다 보이는 사건**만 반환하라고 요청했습니다. 프레임을 보고 판단할 모델에는 사람의 2회 라벨을 전달하지 않았습니다.


In [ ]:
for item in review["videos"]:
    video_id=item["video_id"]
    baseline=json.loads((evidence/f"{video_id}-baseline.json").read_text())
    targeted=json.loads((evidence/f"{video_id}-transition-eval.json").read_text())
    print("\n",video_id,"일반 추출기:")
    for e in baseline["events"]: print(e["start_time"],e["end_time"],e["action"])
    print("전환 모델 원시 행:")
    for window in targeted["results"]:
        for e in window["response"].get("events",[]):
            print(e["start_time"],e["end_time"],e["status"],e["subject_description"])


In [ ]:
# 개체별 사건 로그를 사용합니다. 시각 차이만으로 합치지 않습니다.
full_audit=json.loads((evidence/"state-transition-audit.json").read_text(encoding="utf-8"))
for case in full_audit["cases"]:
    if case["video_id"] not in ("smartbench_0518", "smartbench_0642"):
        continue
    counts={s:sum(r["status"]==s for r in case["candidates"])
            for s in ("confirmed","duplicate","uncertain","rejected")}
    print(case["video_id"], "모델 원시", case["model"]["raw_rows"],
          "재감사 확정", counts["confirmed"], "중복", counts["duplicate"])
    for row in case["candidates"]:
        print(" ", row["time"], row["subject"], row["episode"], row["status"])


## 해석

`0518`에서는 두 강아지가 각각 바닥에서 소파로 올라간 전환이 보여 2회입니다. `0642`에서는 두 번째 강아지의 입수를 모델이 시간창 양쪽에서 두 번 기록했지만, 같은 개체가 물 밖으로 나왔다가 다시 들어간 장면은 확인되지 않습니다. 따라서 모델 원시 3행을 **2회 확정 + 1행 중복**으로 재감사했습니다. 시간 간격만으로 사건을 합치지 않습니다.

여기서 재감사 판정은 사람이 저장된 출력과 프레임을 대조한 결과입니다. 자동 파이프라인의 보편적 정확도를 뜻하지 않습니다. 나머지 앞선 영상도 [state-transition-audit.ipynb](smarthome-bench-state-transition-audit.ipynb)에 같은 기준으로 기록했습니다.
